<a class="anchor" id="import"></a>
# <p style="padding:10px;background-color:#800080;margin:0;color:white;font-family:newtimeroman;font-size:150%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500"> S3E20 Notebook: Predict CO2 Emissions in Rwanda </p> 

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#006600; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300"> 1 | Introduction </p>

<div style="border-radius:10px; border:#006600 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
In this competitition, we are tasked to predict 2022 CO2 emissions in 497 different locations in Africa. In the training data, we have the CO2 emissions for the years 2019-2021
    
**Contents of this Notebook:**
    
1. Remove the one-off COVID trend in 2020 via smoothing. alternatively, imputing 2020 with the mean of 2019 and 2021 is a valid approach as well, but was not implemented here
    
2. Observe that locations closer to the location with maximum emission have high emission levels as well. Perform a K-Means clustering to cluster the data points based on their location. This allows the data points with similar emissions to be grouped together
    
3. Experiment with some ensemble models to test out their CV on 2021 data, given 2019 and 2020 as train data
    
4. 😆 Submit the non-ML lazy prediction instead of the ensembled prediction because it gives a better score

In [ ]:
#https://www.kaggle.com/competitions/playground-series-s3e14/discussion/410627

!wget http://bit.ly/3ZLyF82 -O CSS.css -q
    
from IPython.core.display import HTML
with open('./CSS.css', 'r') as file:
    custom_css = file.read()

HTML(custom_css)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from tqdm import tqdm
from sklearn.preprocessing import SplineTransformer
from holidays import CountryHoliday
from tqdm.notebook import tqdm
from typing import List

#https://www.kaggle.com/code/kimtaehun/multi-label-classification-with-complete-eda
from colorama import Style, Fore, Back
## Set Plot Parameters       
plt.style.use("Solarize_Light2")
color_pal = ["#4CAF50", "#780060", "#FFBF00", "#6495ED", "#DE3163",
             "#922710", "#C99BE8", "#FF8700", "#E0FF00", "#2E00FF"]

rc = {
    "axes.facecolor": "#FFF9ED",
    "figure.facecolor": "#FFF9ED",
    "axes.edgecolor": "#000000",
    "grid.color": "#EBEBE7",
    "font.family": "serif",
    "axes.labelcolor": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "grid.alpha": 0.4
}

sns.set(rc=rc)

blk = Style.BRIGHT + Fore.BLACK
red = Style.BRIGHT + Fore.RED
gld = Style.BRIGHT + Fore.YELLOW
blu = Style.BRIGHT + Fore.BLUE
res = Style.RESET_ALL

from category_encoders import OneHotEncoder, MEstimateEncoder, GLMMEncoder, OrdinalEncoder
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold, RepeatedKFold, TimeSeriesSplit, train_test_split, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor, LogisticRegression
from sklearn.linear_model import PassiveAggressiveRegressor, ARDRegression
from sklearn.linear_model import TheilSenRegressor, RANSACRegressor, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score, roc_curve
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, LabelEncoder, SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_selection import RFECV
from sklearn.decomposition import PCA
from xgboost import XGBRegressor, XGBClassifier
import lightgbm as lgbm
from lightgbm import LGBMRegressor, LGBMClassifier
from lightgbm import log_evaluation, early_stopping, record_evaluation
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from sklearn import set_config
from sklearn.multioutput import MultiOutputClassifier
from datetime import datetime, timedelta
import gc

import warnings
warnings.filterwarnings('ignore')

set_config(transform_output = 'pandas')

pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)

In [ ]:
M = 1.07

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#FF0000; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #FF0000"> 2 | Examine Data </p>

<div style="border-radius:10px; border:#FF0000 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**2.1**
    
Here we are trying to smooth the 2020 data to remove the covid trends
    
1. Using the smoothed imported dataset
2. Using the mean of 2019 and 2021 values [https://www.kaggle.com/code/kacperrabczewski/rwanda-co2-step-by-step-guide]

In [ ]:
extrp = pd.read_csv("/kaggle/input/ps3e20-covid-updated/PS3E20_train_covid_updated")
extrp = extrp[(extrp["year"] == 2020)]

In [ ]:
extrp

In [ ]:
DATA_DIR = "/kaggle/input/playground-series-s3e20/"
train = pd.read_csv(DATA_DIR + "train.csv")
test = pd.read_csv(DATA_DIR + "test.csv")

def add_features(df):
    #df["week"] = df["year"].astype(str) + "-" + df["week_no"].astype(str)
    #df["date"] = df["week"].apply(lambda x: get_date_from_week_string(x))
    #df = df.drop(columns = ["week"])
    df["week"] = (df["year"] - 2019) * 53 + df["week_no"]
    #df["lat_long"] = df["latitude"].astype(str) + "#" + df["longitude"].astype(str)
    return df

train = add_features(train)
test = add_features(test)

<div style="border-radius:10px; border:#FF0000 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**2.2**
       
Some risky postprocessing on the predictions. 
    
Assuming MAX = max(2019 emissions, 2020 emissions, 2021 emissions) for a data point. 
    
We assign MAX * 1.07 to the prediction if its 2021 emission > 2019 emission, else we just assign MAX. Reference: https://www.kaggle.com/competitions/playground-series-s3e20/discussion/430152

In [ ]:
vals = set()
for x in train[["latitude", "longitude"]].values:
    vals.add(tuple(x))
    
vals = list(vals)

In [ ]:
zeros = []

for lat, long in vals:
    subset = train[(train["latitude"] == lat) & (train["longitude"] == long)]
    em_vals = subset["emission"].values
    if all(x == 0 for x in em_vals):
        zeros.append([lat, long])

In [ ]:
test["2021_emission"] = test["week_no"]
test["2020_emission"] = test["week_no"]
test["2019_emission"] = test["week_no"]

for lat, long in vals:
    test.loc[(test.latitude == lat) & (test.longitude == long), "2021_emission"] = train.loc[(train.latitude == lat) & (train.longitude == long) & (train.year == 2021) & (train.week_no <= 48), "emission"].values
    test.loc[(test.latitude == lat) & (test.longitude == long), "2020_emission"] = train.loc[(train.latitude == lat) & (train.longitude == long) & (train.year == 2020) & (train.week_no <= 48), "emission"].values
    test.loc[(test.latitude == lat) & (test.longitude == long), "2019_emission"] = train.loc[(train.latitude == lat) & (train.longitude == long) & (train.year == 2019) & (train.week_no <= 48), "emission"].values
    #print(train.loc[(train.latitude == lat) & (train.longitude == long) & (train.year == 2021), "emission"])
    
test["ratio"] = (test["2021_emission"] / test["2019_emission"]).replace(np.nan, 0)
test["pos_ratio"] = test["ratio"].apply(lambda x: max(x, 1))
test["pos_ratio"] = test["pos_ratio"].apply(lambda x: 1.07 if x > 1 else x)
test["max"] = test[["2019_emission", "2020_emission", "2021_emission"]].max(axis=1)
test["lazy_pred"] = test["max"] * test["pos_ratio"]
test = test.drop(columns = ["ratio", "pos_ratio", "max", "2019_emission", "2020_emission", "2021_emission"])

In [ ]:
train.loc[train.year == 2020, "emission"] = extrp

In [ ]:
train

In [ ]:
test

<div style="border-radius:10px; border:#FF0000 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**Insights**
    
The train dataset has 79023 observations and the test dataset has 24353 observations. As we observe, some columns have null values

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#800080; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #800080"> 3 | EDA and Data Distribution </p>

In [ ]:
#beautify dataframe: reference https://www.kaggle.com/code/tetsutani/ps3e15-eda-ensemble-and-stacking-baseline
def set_frame_style(df, caption=""):
    """Helper function to set dataframe presentation style.
    """
    return df.style.background_gradient(cmap='Blues').set_caption(caption).set_table_styles([{
    'selector': 'caption',
    'props': [
        ('color', 'Blue'),
        ('font-size', '18px'),
        ('font-weight','bold')
    ]}])

#https://www.kaggle.com/code/mohammadrazeghi/p03e15-preprocess-pipeline-baseline-xgboost
def summary(df):
    print(f'{Style.BRIGHT}{Fore.BLACK}Head of the dataset \n \n {"*"*100}')
    display(df.head(3))
    print(f'\n {Style.BRIGHT}{Fore.BLACK}{"*"*100}\n')
    print(f"{Style.BRIGHT}{Fore.BLACK}Summary of the dataset ---->  dataset has {Fore.RED}{df.shape[1]-1}{Fore.BLACK} features and {Fore.RED}{df.shape[0]}{Fore.BLACK} examples.")
    print(f'\n{Style.BRIGHT}{Fore.BLACK}{"*"*100}\n{Style.RESET_ALL}')
    summary = pd.DataFrame(index=df.columns)
    desc = pd.DataFrame(df.describe(include='all').transpose())
    summary["Count"] = desc['count'].values
    summary["Unique"] = df.nunique().values
    summary["Missing"] = df.isnull().sum().values
    summary["Duplicated"] = df.duplicated().sum()
    summary['Std'] = desc['std'].values
    summary["Mode"] = df.mode().values[0]
    summary["Median"] = df.median()
    summary['Mean'] = desc['mean'].values
    summary['Min'] = desc['min'].values
    summary['Max'] = desc['max'].values
    summary["First Value"] = df.loc[0].values
    summary["Last Value"] = df.loc[df.shape[0]-1].values
    summary["Types"] = df.dtypes
    return display(set_frame_style(summary, "Summary Statistics"))

In [ ]:
summary(train)

In [ ]:
summary(test)

In [ ]:
def plot_emission(train):
    
    plt.figure(figsize=(15, 6))
    sns.lineplot(data=train, x="week", y="emission", label="Emission", alpha=0.7, color='blue')

    plt.xlabel('Week')
    plt.ylabel('Emission')
    plt.title('Emission over time')

    plt.legend()
    plt.tight_layout()
    plt.show()
    
plot_emission(train)

In [ ]:
sns.histplot(train["emission"])

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#2e3ca5; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #2e3ca5"> 4 | Data Transformation </p>


In [ ]:
print(len(vals))

<div style="border-radius:10px; border:#2e3ca5 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**Insights**
    
There are 497 unique latitude-longtitude combinations

<div style="border-radius:10px; border:#2e3ca5 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**4.1**
    
Most of the features are just noise, we can remove them. (Reference: multiple discussion posts)

In [ ]:
#train = train.drop(columns = ["ID_LAT_LON_YEAR_WEEK", "lat_long"])
#test = test.drop(columns = ["ID_LAT_LON_YEAR_WEEK", "lat_long"])

train = train[["latitude", "longitude", "year", "week_no", "emission"]]
test = test[["latitude", "longitude", "year", "week_no", "lazy_pred"]]

<div style="border-radius:10px; border:#2e3ca5 solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**4.2**
    
K Means Clustering + Distance to highest emission

In [ ]:
#https://www.kaggle.com/code/lucasboesen/simple-catboost-6-features-cv-21-7
from sklearn.cluster import KMeans
import haversine as hs

km_train = train.groupby(by=['latitude', 'longitude'], as_index=False)['emission'].mean()
model = KMeans(n_clusters = 7, random_state = 42)
model.fit(km_train)
yhat_train = model.predict(km_train)
km_train['kmeans_group'] = yhat_train

""" Own Groups """
# Some locations have emission == 0
km_train['is_zero'] = km_train['emission'].apply(lambda x: 'no_emission_recorded' if x==0 else 'emission_recorded')

# Distance to the highest emission location
max_lat_lon_emission = km_train.loc[km_train['emission']==km_train['emission'].max(), ['latitude', 'longitude']]
km_train['distance_to_max_emission'] = km_train.apply(lambda x: hs.haversine((x['latitude'], x['longitude']), (max_lat_lon_emission['latitude'].values[0], max_lat_lon_emission['longitude'].values[0])), axis=1)

train = train.merge(km_train[['latitude', 'longitude', 'kmeans_group', 'distance_to_max_emission']], on=['latitude', 'longitude'])
test = test.merge(km_train[['latitude', 'longitude', 'kmeans_group', 'distance_to_max_emission']], on=['latitude', 'longitude'])
#train = train.drop(columns = ["latitude", "longitude"])
#test = test.drop(columns = ["latitude", "longitude"])

In [ ]:
train

In [ ]:
test

In [ ]:
cat_params = {
    
    'n_estimators': 799, 
    'learning_rate': 0.09180872710592884,
    'depth': 8, 
    'l2_leaf_reg': 1.0242996861886846, 
    'subsample': 0.38227256755249117, 
    'colsample_bylevel': 0.7183481537623551,
    'random_state': 42,
    "silent": True,
}

lgb_params = {
    
    'n_estimators': 835, 
    'max_depth': 12, 
    'reg_alpha': 3.849279869880706, 
    'reg_lambda': 0.6840221712299135, 
    'min_child_samples': 10, 
    'subsample': 0.6810493885301987, 
    'learning_rate': 0.0916362259866008, 
    'colsample_bytree': 0.3133780298325982, 
    'colsample_bynode': 0.7966712089198238,
    "random_state": 42,
}

xgb_params = {
    
    "random_state": 42,
}

rf_params = {
    
    'n_estimators': 263, 
    'max_depth': 41, 
    'min_samples_split': 10, 
    'min_samples_leaf': 3,
    "random_state": 42,
    "verbose": 0
}

et_params = {
    
    "random_state": 42,
    "verbose": 0
}

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#8B8000; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #8B8000"> 5 | Validate Performance on 2021 data </p>

In [ ]:
def rmse(a, b):
    return mean_squared_error(a, b, squared=False)

In [ ]:
validation = train[train.year == 2021]
clusters = train["kmeans_group"].unique()

for i in range(len(clusters)):
               
    cluster = clusters[i]
    
    print("==============================================")
    print(f"{Fore.GREEN} Cluster {cluster} {Fore.BLACK}")
    
    
    train_c = train[train["kmeans_group"] == cluster]
    
    X_train = train_c[train_c.year < 2021].drop(columns = ["emission", "kmeans_group"])
    y_train = train_c[train_c.year < 2021]["emission"].copy()
    X_val = train_c[train_c.year >= 2021].drop(columns = ["emission", "kmeans_group"])
    y_val = train_c[train_c.year >= 2021]["emission"].copy()
    
    
    
    #=======================================================================================
    catboost_reg = CatBoostRegressor(**cat_params)
    catboost_reg.fit(X_train, y_train, eval_set=(X_val, y_val))

    catboost_pred = catboost_reg.predict(X_val) * M
    print(f"RMSE of CatBoost: {rmse(catboost_pred, y_val)}")

    #=======================================================================================
    lightgbm_reg = LGBMRegressor(**lgb_params)
    lightgbm_reg.fit(X_train, y_train, eval_set=(X_val, y_val), verbose = -1)

    lightgbm_pred = lightgbm_reg.predict(X_val) * M
    print(f"RMSE of LightGBM: {rmse(lightgbm_pred, y_val)}")

    #=======================================================================================
    xgb_reg = XGBRegressor(**xgb_params)
    xgb_reg.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose = False)

    xgb_pred = xgb_reg.predict(X_val) * M
    print(f"RMSE of XGBoost: {rmse(xgb_pred, y_val)}")

    #=======================================================================================
    rf_reg = RandomForestRegressor(**rf_params)
    rf_reg.fit(X_train, y_train)

    rf_pred = rf_reg.predict(X_val) * M
    print(f"RMSE of Random Forest: {rmse(rf_pred, y_val)}")

    #=======================================================================================
    et_reg = ExtraTreesRegressor(**et_params)
    et_reg.fit(X_train, y_train)

    et_pred = et_reg.predict(X_val) * M
    print(f"RMSE of Extra Trees: {rmse(et_pred, y_val)}")
    
    
    overall_pred = lightgbm_pred #(catboost_pred + lightgbm_pred) / 2
    validation.loc[validation["kmeans_group"] == cluster, "emission"] = overall_pred
    
    print(f"RMSE Overall: {rmse(overall_pred, y_val)}")

print("==============================================")
print(f"[DONE] RMSE of all clusters: {Fore.BLUE}{rmse(validation['emission'], train[train.year == 2021]['emission'])}{Fore.BLACK}")
print(f"[DONE] RMSE of all clusters Week 1-20: {Fore.BLUE}{rmse(validation[validation.week_no < 21]['emission'], train[(train.year == 2021) & (train.week_no < 21)]['emission'])}{Fore.BLACK}")
print(f"[DONE] RMSE of all clusters Week 21+: {Fore.BLUE}{rmse(validation[validation.week_no >= 21]['emission'], train[(train.year == 2021) & (train.week_no  >= 21)]['emission'])}{Fore.BLACK}")

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#FFC0CB; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #FFC0CB"> 6 | Predicting 2022 result</p>

In [ ]:
clusters = train["kmeans_group"].unique()

for i in tqdm(range(len(clusters))):
    
    cluster = clusters[i]
    
    train_c = train[train["kmeans_group"] == cluster]
    if "emission" in test.columns:
        test_c = test[test["kmeans_group"] == cluster].drop(columns = ["emission", "kmeans_group", "lazy_pred"])
    else:
        test_c = test[test["kmeans_group"] == cluster].drop(columns = ["kmeans_group", "lazy_pred"])
    
    X = train_c.drop(columns = ["emission", "kmeans_group"])
    y = train_c["emission"].copy()
    #=======================================================================================
    catboost_reg = CatBoostRegressor(**cat_params)
    catboost_reg.fit(X, y)
    #print(test_c)

    catboost_pred = catboost_reg.predict(test_c)

    #=======================================================================================
    lightgbm_reg = LGBMRegressor(**lgb_params)
    lightgbm_reg.fit(X, y, verbose = -1)
    #print(test_c)

    lightgbm_pred = lightgbm_reg.predict(test_c)

    #=======================================================================================
    #xgb_reg = XGBRegressor(**xgb_params)
    #xgb_reg.fit(X, y, verbose = False)

    #xgb_pred = xgb_reg.predict(test)

    #=======================================================================================
    rf_reg = RandomForestRegressor(**rf_params)
    rf_reg.fit(X, y)

    rf_pred = rf_reg.predict(test_c)

    #=======================================================================================
    #et_reg = ExtraTreesRegressor(**et_params)
    #et_reg.fit(X, y)

    #et_pred = et_reg.predict(test)

    overall_pred = lightgbm_pred #(catboost_pred + lightgbm_pred) / 2
    test.loc[test["kmeans_group"] == cluster, "emission"] = overall_pred

In [ ]:
zeros

In [ ]:
test["emission"] = test["emission"] * 1.07

<div style="border-radius:10px; border:#FFC0CB solid; padding: 15px; background-color: #F3f9ed; font-size:100%; text-align:left">
    
**6.1**
    
This is a guess, that the emission will continue to remain at 0 in 2022, if they have been 0 throughout 2019-2021

In [ ]:
sub = pd.read_csv(DATA_DIR + 'sample_submission.csv')
backup = pd.read_csv("/kaggle/input/ps3e20-ensembling-with-score-nudge/submission.csv")
backup2 = pd.read_csv("/kaggle/input/emission-rmse-randomforest/submission.csv")
test["emission"] = (test["emission"] + backup["emission"] + backup2["emission"]) / 3

for lat, long in zeros:
    test.loc[(test["latitude"] == lat) & (test["longitude"] == long), "emission"] = 0
    
#https://www.kaggle.com/competitions/playground-series-s3e20/discussion/429717
test.loc[test['longitude']==29.321, 'emission'] = train.loc[(train['year']==2021)&(train['week_no']<=48)&(train['longitude']==29.321),'emission'].values

sub["emission"] = test["emission"].apply(lambda x: max(x, 0))

In [ ]:
sub.to_csv('submission.csv', index=False)

sub